# 编程练习：使用 Neural Networks 识别手写数字（二分类）

本练习将使用 neural network 识别手写数字 0 和 1。

## 目录

- [1 - 导入依赖库](#1)
- [2 - Neural Networks](#2)
  - [2.1 问题描述](#2.1)
  - [2.2 数据集](#2.2)
  - [2.3 模型表示](#2.3)
  - [2.4 使用 TensorFlow 实现模型](#2.4)
    - [练习 1](#ex01)
  - [2.5 使用 NumPy 实现 Forward Propagation](#2.5)
    - [练习 2](#ex02)
  - [2.6 Vectorized NumPy 实现（可选）](#2.6)
    - [练习 3](#ex03)
  - [2.7 恭喜完成实验](#2.7)
  - [2.8 NumPy Broadcasting 教程（可选）](#2.8)


<a name="1"></a>
## 1 - 导入依赖库

首先运行下面的代码单元，导入本作业所需的库。

- [NumPy](https://numpy.org/)：Python 的基础科学计算库。
- [Matplotlib](http://matplotlib.org)：常用的 Python 绘图库。
- [TensorFlow](https://www.tensorflow.org/)：常用的 Machine Learning 平台。


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
from autils import *
%matplotlib inline

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

**TensorFlow 与 Keras**

TensorFlow 是 Google 开发的 Machine Learning 库。2019 年，Google 在发布 TensorFlow 2.0 时整合了 Keras。Keras 最初由 François Chollet 独立开发，提供了简洁、以 layer 为中心的接口。本课程使用 TensorFlow 中的 Keras 接口。


<a name="2"></a>
## 2 - Neural Networks

在 Course 1 中，你实现了 logistic regression，并通过构造 polynomial features 处理非线性 decision boundary。对于图像识别等更复杂的任务，可以使用 neural network。

<a name="2.1"></a>
### 2.1 问题描述

本练习使用 neural network 识别手写数字 0 和 1，属于 binary classification。手写数字识别广泛用于读取信封邮政编码、银行支票金额等任务。未来的作业中，你将把网络扩展为识别全部 10 个数字（0–9）。

本练习将展示如何将已有知识应用于这一 classification 任务。

<a name="2.2"></a>
### 2.2 数据集

下面的 `load_data()` 将数据加载到 `X` 和 `y`。

- 数据集包含 1000 个手写数字样本，仅包含数字 0 和 1。
- 每个样本是一张 $20\times20$ 的 grayscale image（灰度图像），每个像素由一个表示灰度强度的浮点数表示。
- 将每张图像展开为 400 维 vector，并放入 `X` 的一行，因此 `X` 的 shape 为 `(1000, 400)`。

$$
X=
\begin{pmatrix}
\text{--- }(\mathbf{x}^{(1)})^T\text{ ---}\\
\text{--- }(\mathbf{x}^{(2)})^T\text{ ---}\\
\vdots\\
\text{--- }(\mathbf{x}^{(m)})^T\text{ ---}
\end{pmatrix}
$$

Label vector `y` 的 shape 为 `(1000, 1)`：图像为数字 0 时，`y = 0`；图像为数字 1 时，`y = 1`。

本数据集是 [MNIST 手写数字数据集](http://yann.lecun.com/exdb/mnist/)的一个子集。


In [ ]:
# load dataset
X, y = load_data()

<a name="toc_89367_2.2.1"></a>
#### 2.2.1 查看变量

先打印变量，熟悉其内容。下面显示 `X` 和 `y` 的部分元素。


In [ ]:
print ('The first element of X is: ', X[0])

In [ ]:
print ('The first element of y is: ', y[0,0])
print ('The last element of y is: ', y[-1,0])

<a name="toc_89367_2.2.2"></a>
#### 2.2.2 检查变量维度

打印 `X` 和 `y` 的 shape，确认数据集中包含多少训练样本。


In [ ]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

<a name="toc_89367_2.2.3"></a>
#### 2.2.3 可视化数据

下面随机选择 `X` 的 64 行，将每行还原为 $20\times20$ 的灰度图像并一起展示。每张图像上方显示其 label。


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(8,8))
fig.tight_layout(pad=0.1)

for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Display the label above the image
    ax.set_title(y[random_index,0])
    ax.set_axis_off()

<a name="2.3"></a>
### 2.3 模型表示

本作业的 neural network 如下图所示，包含 3 个使用 sigmoid activation 的 dense layer。输入是图像像素值，每张图像有 $20\times20=400$ 个输入。

<img src="../images/C2_W1_Assign1.PNG" alt="手写数字二分类网络结构" width="500" height="400">


网络的第一层有 25 个 unit，第二层有 15 个 unit，第三层有 1 个输出 unit。

对于输入数量为 $s_{in}$、输出 unit 数量为 $s_{out}$ 的 dense layer，$W$ 的 shape 为 $(s_{in},s_{out})$，$b$ 包含 $s_{out}$ 个元素。因此：

| Layer | Weight shape | Bias shape |
| --- | --- | --- |
| 1 | `W1: (400, 25)` | `b1: (25,)` |
| 2 | `W2: (25, 15)` | `b2: (15,)` |
| 3 | `W3: (15, 1)` | `b3: (1,)` |

> **注意：** Bias vector 可以用 1-D `(n,)` 或 2-D `(1, n)` array 表示。TensorFlow 使用 1-D 表示，本实验也遵循这一约定。


<a name="2.4"></a>
### 2.4 使用 TensorFlow 实现模型


TensorFlow 模型按 layer 构建。你指定每层的*输出维度*，框架据此计算下一层的输入维度。第一层的输入维度可以在 `model.fit` 接收数据时推断。

也可以显式指定输入，例如：

```python
tf.keras.Input(shape=(400,))
```

本实验采用显式输入，以便提前查看参数的 shape。


<a name="ex01"></a>
#### 练习 1

使用 Keras 的 [Sequential model](https://keras.io/guides/sequential_model/) 和 [Dense layer](https://keras.io/api/layers/core_layers/dense/)，配合 sigmoid activation，构建上述网络。


In [ ]:
# UNQ_C1
# GRADED CELL: Sequential model

model = Sequential(
    [               
        tf.keras.Input(shape=(400,)),    #specify input size
        ### START CODE HERE ### 
        
        ### END CODE HERE ### 
    ], name = "my_model" 
)                            


In [ ]:
model.summary()

<details>
<summary><strong>预期输出（点击展开）</strong></summary>

`model.summary()` 显示模型摘要。由于已指定输入大小，weight 和 bias 的 shape 已确定，因此可以显示每层的参数总数。Layer 名称可能因自动命名而不同。

```
Model: "my_model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
dense (Dense)                (None, 25)                10025     
_________________________________________________________________
dense_1 (Dense)              (None, 15)                390       
_________________________________________________________________
dense_2 (Dense)              (None, 1)                 16        
=================================================================
Total params: 10,431
Trainable params: 10,431
Non-trainable params: 0
_________________________________________________________________
```

</details>


<details>
<summary><strong>点击查看提示</strong></summary>

按照课程介绍，可以写成：

```python
model = Sequential(                      
    [                                   
        tf.keras.Input(shape=(400,)),    # specify input size (optional)
        Dense(25, activation='sigmoid'), 
        Dense(15, activation='sigmoid'), 
        Dense(1,  activation='sigmoid')  
    ], name = "my_model"                                    
)
```

</details>


In [ ]:
# UNIT TESTS
from public_tests import * 

test_c1(model)

摘要中的参数数量，对应各个 weight 和 bias array 的元素总数，如下所示。


In [ ]:
L1_num_params = 400 * 25 + 25  # W1 parameters  + b1 parameters
L2_num_params = 25 * 15 + 15   # W2 parameters  + b2 parameters
L3_num_params = 15 * 1 + 1     # W3 parameters  + b3 parameters
print("L1 params = ", L1_num_params, ", L2 params = ", L2_num_params, ",  L3 params = ", L3_num_params )

下面进一步检查 weight，确认 TensorFlow 创建的参数 shape 与前面的计算一致。


In [ ]:
[layer1, layer2, layer3] = model.layers

In [ ]:
#### Examine Weights shapes
W1,b1 = layer1.get_weights()
W2,b2 = layer2.get_weights()
W3,b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")

**预期输出：**

```
W1 shape = (400, 25), b1 shape = (25,)  
W2 shape = (25, 15), b2 shape = (15,)  
W3 shape = (15, 1), b3 shape = (1,)
```


`get_weights()` 返回由 NumPy array 组成的列表。也可以直接访问 tensor 形式的参数。请观察最后一层参数 tensor 的 shape。


In [ ]:
print(model.layers[2].weights)

下面的代码定义 loss function，并运行基于 gradient 的优化，使模型参数拟合训练数据。下周将更详细地介绍这一过程。


In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X,y,
    epochs=20
)

使用 [Keras 的 `predict`](https://www.tensorflow.org/api_docs/python/tf/keras/Model) 对样本进行预测。`predict` 的输入是批量样本 array，因此单个样本也需要调整为二维。


In [ ]:
prediction = model.predict(X[0].reshape(1,400))  # a zero
print(f" predicting a zero: {prediction}")
prediction = model.predict(X[500].reshape(1,400))  # a one
print(f" predicting a one:  {prediction}")

模型输出可解释为输入图像属于数字 1 的概率。第一个样本是 0，输出应接近 0；第二个样本是 1，输出应接近 1。

与 logistic regression 一样，比较预测概率与 threshold，即可得到最终类别。


In [ ]:
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print(f"prediction after threshold: {yhat}")

下面随机选择 64 个数字，比较预测结果与 label。运行可能需要一点时间。


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(8,8))
fig.tight_layout(pad=0.1,rect=[0, 0.03, 1, 0.92]) #[left, bottom, right, top]

for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Predict using the Neural Network
    prediction = model.predict(X[random_index].reshape(1,400))
    if prediction >= 0.5:
        yhat = 1
    else:
        yhat = 0
    
    # Display the label above the image
    ax.set_title(f"{y[random_index,0]},{yhat}")
    ax.set_axis_off()
fig.suptitle("Label, yhat", fontsize=16)
plt.show()

<a name="2.5"></a>
### 2.5 使用 NumPy 实现 Forward Propagation

如课程所述，可以使用 NumPy 自行实现 dense layer，再用它构建多层 neural network。

<img src="../images/C2_W1_dense2.PNG" alt="Dense layer 的 NumPy 计算过程" width="600" height="450">


<a name="ex02"></a>
#### 练习 2

编写 dense layer 函数：使用 `for` loop 遍历每个 unit `j`，计算输入与 `W[:, j]` 的 dot product，再加上 `b[j]` 得到 `z`，最后计算 activation `g(z)`。

本节先采用循环实现；可选课程中的 matrix 运算将在后面的部分使用。


In [ ]:
# UNQ_C2
# GRADED FUNCTION: my_dense

def my_dense(a_in, W, b, g):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      a_out (ndarray (j,))  : j units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
### START CODE HERE ### 
        
### END CODE HERE ### 
    return(a_out)


In [ ]:
# Quick Check
x_tst = 0.1*np.arange(1,3,1).reshape(2,)  # (1 examples, 2 features)
W_tst = 0.1*np.arange(1,7,1).reshape(2,3) # (2 input features, 3 output features)
b_tst = 0.1*np.arange(1,4,1).reshape(3,)  # (3 features)
A_tst = my_dense(x_tst, W_tst, b_tst, sigmoid)
print(A_tst)

**预期输出：**

```
[0.54735762 0.57932425 0.61063923]
```


<details>
<summary><strong>点击查看提示</strong></summary>

根据课程内容，函数框架如下：

```python
def my_dense(a_in, W, b, g):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      a_out (ndarray (j,))  : j units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):             
        w =                            # Select weights for unit j. These are in column j of W
        z =                            # dot product of w and a_in + b
        a_out[j] =                     # apply activation to z
    return(a_out)
```

<details>
<summary><strong>查看更多提示</strong></summary>

```python
def my_dense(a_in, W, b, g):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example 
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      a_out (ndarray (j,))  : j units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):             
        w = W[:,j]                     
        z = np.dot(w, a_in) + b[j]     
        a_out[j] = g(z)                
    return(a_out)
```

</details>

</details>


In [ ]:
# UNIT TESTS
test_c2(my_dense)

下面使用 `my_dense` 构建一个三层 neural network。


In [ ]:
def my_sequential(x, W1, b1, W2, b2, W3, b3):
    a1 = my_dense(x,  W1, b1, sigmoid)
    a2 = my_dense(a1, W2, b2, sigmoid)
    a3 = my_dense(a2, W3, b3, sigmoid)
    return(a3)

可以直接复制 TensorFlow 模型训练得到的 weight 和 bias。


In [ ]:
W1_tmp,b1_tmp = layer1.get_weights()
W2_tmp,b2_tmp = layer2.get_weights()
W3_tmp,b3_tmp = layer3.get_weights()

In [ ]:
# make predictions
prediction = my_sequential(X[0], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print( "yhat = ", yhat, " label= ", y[0,0])
prediction = my_sequential(X[500], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print( "yhat = ", yhat, " label= ", y[500,0])

运行下面的单元，比较 NumPy 模型和 TensorFlow 模型的预测。运行可能需要一点时间。


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(8,8))
fig.tight_layout(pad=0.1,rect=[0, 0.03, 1, 0.92]) #[left, bottom, right, top]

for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')

    # Predict using the Neural Network implemented in Numpy
    my_prediction = my_sequential(X[random_index], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
    my_yhat = int(my_prediction >= 0.5)

    # Predict using the Neural Network implemented in Tensorflow
    tf_prediction = model.predict(X[random_index].reshape(1,400))
    tf_yhat = int(tf_prediction >= 0.5)
    
    # Display the label above the image
    ax.set_title(f"{y[random_index,0]},{tf_yhat},{my_yhat}")
    ax.set_axis_off() 
fig.suptitle("Label, yhat Tensorflow, yhat Numpy", fontsize=16)
plt.show()

<a name="2.6"></a>
### 2.6 Vectorized NumPy 实现（可选）

可选课程介绍了如何使用 vector 和 matrix 运算加快计算。下图展示了对一个样本同时计算某一层全部 unit 输出的方法。

<img src="../images/C2_W1_VectorMatrix.PNG" alt="使用 vector-matrix 运算计算一个 layer" width="600" height="450">

下面使用 `X` 中的样本和参数 `W1`、`b1` 演示。`np.matmul` 用于 matrix multiplication，输入与 `W` 的维度必须像上图一样匹配。


In [ ]:
x = X[0].reshape(-1,1)         # column vector (400,1)
z1 = np.matmul(x.T,W1) + b1    # (1,400)(400,25) = (1,25)
a1 = sigmoid(z1)
print(a1.shape)

还可以进一步通过一次 matrix-matrix 运算，计算全部样本在全部 unit 上的结果。

<img src="../images/C2_W1_MatrixMatrix.PNG" alt="批量样本的 matrix-matrix 运算" width="600" height="450">

完整计算为 $\mathbf{Z}=\mathbf{XW}+\mathbf{b}$。NumPy broadcasting 会将 $\mathbf{b}$ 按行应用于 $m$ 个样本。如果你尚不熟悉 broadcasting，可以阅读 Notebook 末尾的简短教程。


<a name="ex03"></a>
#### 练习 3

编写 `my_dense_v`，使用 `np.matmul()` 对包含多个样本的 matrix 执行 layer 计算。


In [ ]:
# UNQ_C3
# GRADED FUNCTION: my_dense_v

def my_dense_v(A_in, W, b, g):
    """
    Computes dense layer
    Args:
      A_in (ndarray (m,n)) : Data, m examples, n features each
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j,1)) : bias vector, j units  
      g    activation function (e.g. sigmoid, relu..)
    Returns
      A_out (ndarray (m,j)) : m examples, j units
    """
### START CODE HERE ### 
    
    
### END CODE HERE ### 
    return(A_out)

In [ ]:
X_tst = 0.1*np.arange(1,9,1).reshape(4,2) # (4 examples, 2 features)
W_tst = 0.1*np.arange(1,7,1).reshape(2,3) # (2 input features, 3 output features)
b_tst = 0.1*np.arange(1,4,1).reshape(1,3) # (3 features, 1)
A_tst = my_dense_v(X_tst, W_tst, b_tst, sigmoid)
print(A_tst)

**预期输出：**

```
[[0.54735762 0.57932425 0.61063923]
 [0.57199613 0.61301418 0.65248946]
 [0.5962827  0.64565631 0.6921095 ]
 [0.62010643 0.67699586 0.72908792]]
```


<details>
<summary><strong>点击查看提示</strong></summary>

Matrix 形式只需要一两行：先计算 `A_in` 与 `W` 的 matrix product 并加上 `b` 得到 `Z`，再计算 `g(Z)`。

<details>
<summary><strong>点击查看代码</strong></summary>

```python
def my_dense_v(A_in, W, b, g):
    """
    Computes dense layer
    Args:
      A_in (ndarray (m,n)) : Data, m examples, n features each
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j,))  : bias vector, j units
      g                   : activation function (e.g. sigmoid, relu)
    Returns:
      A_out (ndarray (m,j)): m examples, j units
    """
    Z = np.matmul(A_in, W) + b
    A_out = g(Z)
    return A_out
```

</details>

</details>


In [ ]:
# UNIT TESTS
test_c3(my_dense_v)

下面使用 `my_dense_v` 构建一个三层 neural network。


In [ ]:
def my_sequential_v(X, W1, b1, W2, b2, W3, b3):
    A1 = my_dense_v(X,  W1, b1, sigmoid)
    A2 = my_dense_v(A1, W2, b2, sigmoid)
    A3 = my_dense_v(A2, W3, b3, sigmoid)
    return(A3)

再次复制 TensorFlow 模型训练得到的 weight 和 bias。


In [ ]:
W1_tmp,b1_tmp = layer1.get_weights()
W2_tmp,b2_tmp = layer2.get_weights()
W3_tmp,b3_tmp = layer3.get_weights()

下面使用新模型*一次性预测所有样本*。请留意输出的 shape。


In [ ]:
Prediction = my_sequential_v(X, W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp )
Prediction.shape

与之前一样使用 0.5 作为 threshold，这次同时处理全部预测值。


In [ ]:
Yhat = (Prediction >= 0.5).numpy().astype(int)
print("predict a zero: ",Yhat[0], "predict a one: ", Yhat[500])

运行下面的单元，显示刚刚计算的预测结果。运行可能需要一点时间。


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1, rect=[0, 0.03, 1, 0.92]) #[left, bottom, right, top]

for i, ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20, 20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
   
    # Display the label above the image
    ax.set_title(f"{y[random_index,0]}, {Yhat[random_index, 0]}")
    ax.set_axis_off() 
fig.suptitle("Label, Yhat", fontsize=16)
plt.show()

下面查看其中一张被错误分类的图像。


In [ ]:
fig = plt.figure(figsize=(1, 1))
errors = np.where(y != Yhat)
random_index = errors[0][0]
X_random_reshaped = X[random_index].reshape((20, 20)).T
plt.imshow(X_random_reshaped, cmap='gray')
plt.title(f"{y[random_index,0]}, {Yhat[random_index, 0]}")
plt.axis('off')
plt.show()

<a name="2.7"></a>
### 2.7 恭喜完成实验！

你已经成功构建并使用了 neural network。


<a name="2.8"></a>
### 2.8 NumPy Broadcasting 教程（可选）


上面的 $\mathbf{Z}=\mathbf{XW}+\mathbf{b}$ 使用 NumPy broadcasting 扩展 $\mathbf{b}$。

若 $\mathbf{X}$ 和 $\mathbf{W}$ 的 shape 分别为 $(m,j_1)$ 和 $(j_1,j_2)$，则乘积的 shape 为 $(m,j_2)$。要加上 shape 为 $(j_2,)$ 的一维 vector $\mathbf{b}$，NumPy 会在概念上将其扩展为 $(m,j_2)$，再进行 element-wise 加法。下面通过简短教程介绍这一机制。


Broadcasting 适用于 element-wise 运算，其直观含义是将长度为 1 的维度“扩展”，使两个 array 的 shape 匹配。

根据 [NumPy Broadcasting 规则](https://NumPy.org/doc/stable/user/basics.broadcasting.html)，比较 shape 时从最右侧的维度开始向左进行。对应的两个维度满足以下任一条件即可兼容：

- 两者长度相等。
- 其中一个长度为 1。

缺失的前导维度可视为长度 1。如果不满足规则，会出现 `ValueError: operands could not be broadcast together`，表示 shape 不兼容。兼容时，每个结果维度采用对应输入维度中不为 1 的长度。

下面是一些示例。


<figure style="text-align:center;">
  <img src="../images/C2_W1_Assign1_BroadcastIndexes.PNG" alt="计算 broadcasting 结果的 shape" width="400">
  <figcaption>计算 broadcasting 后的 shape。</figcaption>
</figure>


下图说明维度如何扩展，请留意红色文字。


<figure style="text-align:center;">
  <img src="../images/C2_W1_Assign1_Broadcasting.gif" alt="Broadcasting 扩展示意动画" width="600">
  <figcaption>在概念上扩展输入，使其适合 element-wise 运算。</figcaption>
</figure>


上图用“扩展输入后再运算”解释 broadcasting。这是一种概念示意，NumPy 的实际实现会采用更高效的方式，并不一定真的复制数据。

运行下面各个示例前，先猜测结果的 shape。


In [ ]:
a = np.array([1,2,3]).reshape(-1,1)  #(3,1)
b = 5
print(f"(a + b).shape: {(a + b).shape}, \na + b = \n{a + b}")

Broadcasting 同样适用于其他 element-wise 运算：


In [ ]:
a = np.array([1,2,3]).reshape(-1,1)  #(3,1)
b = 5
print(f"(a * b).shape: {(a * b).shape}, \na * b = \n{a * b}")

<figure style="text-align:center;">
  <img src="../images/C2_W1_Assign1_VectorAdd.PNG" alt="Row 与 column 的 element-wise 运算" width="740">
  <figcaption>Row 与 column 之间的 element-wise 运算。</figcaption>
</figure>


In [ ]:
a = np.array([1,2,3,4]).reshape(-1,1)
b = np.array([1,2,3]).reshape(1,-1)
print(a)
print(b)
print(f"(a + b).shape: {(a + b).shape}, \na + b = \n{a + b}")

这正是前面 dense layer 的情形：将 1-D vector $\mathbf{b}$ 加到 shape 为 $(m,j)$ 的 matrix 上。

<figure style="text-align:center;">
  <img src="../images/C2_W1_Assign1_BroadcastMatrix.PNG" alt="Matrix 与 1-D vector 相加" width="740">
  <figcaption>Matrix + 1-D vector。</figcaption>
</figure>
